In [ ]:
import pandas as pd
import numpy as np

In [ ]:
orders_df = pd.read_csv("data/ecommerce_dataset/orders.csv").sort_values("order_date")
products_df = pd.read_csv("data/processed/products_processed.csv").sort_values("updated_at")
order_items_df = pd.read_csv("data/ecommerce_dataset/order_items.csv")

products_df['updated_at'] = pd.to_datetime(products_df['updated_at'])
orders_df['order_date'] = pd.to_datetime(orders_df['order_date'])

In [ ]:
orders_df = orders_df.reset_index(drop=True)
orders_df.head(2)

In [ ]:
products_df = products_df.reset_index(drop=True)
products_df.head(2)

In [ ]:
order_items_df = order_items_df.reset_index(drop=True)
order_items_df.head(2)

In [ ]:
order_items_expanded = order_items_df.merge(
    orders_df[['order_id', 'order_date']],
    on='order_id',
    how='left'
)

order_items_expanded.head(3)

In [ ]:
order_items_expanded = order_items_expanded.sort_values(
    ['order_date', 'order_id']
).reset_index(drop=True)

order_items_expanded.head(3)

In [ ]:
orders_items_processed = pd.merge_asof(
    order_items_expanded,
    products_df,
    left_on="order_date",
    right_on="updated_at",
    by="product_id",
    direction="backward"
)

orders_items_processed['item_price'] = orders_items_processed['price']
orders_items_processed['item_total'] = orders_items_processed['item_price'] * orders_items_processed['quantity']

orders_items_processed.sort_values(by='updated_at', ascending=False).head(3)

In [ ]:
orders_items_processed['item_price'] = orders_items_processed['price']
orders_items_processed['item_total'] = orders_items_processed['item_price'] * orders_items_processed['quantity']

orders_items_processed.sort_values(by='updated_at', ascending=False).head(3)

In [ ]:
orders_items_processed = orders_items_processed.drop(columns=['order_date', 'product_name',
                                                              'category', 'brand', 'price',
                                                              'rating', 'updated_at'])

orders_items_processed.head()

In [ ]:
order_total_amounts = orders_items_processed.groupby('order_id').agg(
    new_total_amount=('item_total', 'sum')
).reset_index()

order_total_amounts.head()

In [ ]:
orders_processed = orders_df.merge(
    order_total_amounts,
    how='left',
    on='order_id'
)

orders_processed.loc[
    ~np.isclose(orders_processed['total_amount'], orders_processed['new_total_amount'], atol=0.01),
    ['order_id', 'total_amount', 'new_total_amount']
]
#orders_processed.sort_values(by='order_date', ascending=False).head()

In [ ]:
orders_processed['total_amount'] = orders_processed['new_total_amount']
orders_processed = orders_processed.drop(columns='new_total_amount')

orders_processed.head()